In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
# load source CSV files
repo_root = Path.cwd()
consti_path = repo_root / "src" / "consti1.csv"
partylist_path = repo_root / "src" / "partylist1.csv"

consti_df = pd.read_csv(consti_path)
partylist_df = pd.read_csv(partylist_path)

consti_df.head(3)

In [ ]:
# precompute Benford data and write to src/benford.json
import math

def normalize_first_digit(series: pd.Series) -> pd.Series:
    digits = pd.to_numeric(series, errors="coerce")
    digits = digits.dropna().astype("int64")
    return digits[(digits >= 1) & (digits <= 9)]

def benford_distribution(digits: pd.Series) -> list[dict]:
    total = int(len(digits))
    counts = digits.value_counts().to_dict()
    rows = []
    for digit in range(1, 10):
        expected_ratio = float(math.log10(1 + 1 / digit))
        rows.append(
            {
                "digit": digit,
                "actual": int(counts.get(digit, 0)),
                "expected_ratio": expected_ratio,
                "expected": float(total * expected_ratio),
                "actual_ratio": float(counts.get(digit, 0) / total) if total > 0 else 0.0,
            }
        )
    return rows

consti_digits = normalize_first_digit(consti_df.get("First_Digit", pd.Series(dtype="float64")))
partylist_digits = normalize_first_digit(partylist_df.get("First_digit", pd.Series(dtype="float64")))

overall_digits = pd.concat([consti_digits, partylist_digits], ignore_index=True)

per_party_digits = []
if "พรรค" in consti_df.columns:
    consti_party = consti_df[["พรรค", "First_Digit"]].copy()
    consti_party["digit"] = pd.to_numeric(consti_party["First_Digit"], errors="coerce")
    consti_party = consti_party.dropna(subset=["พรรค", "digit"])
    consti_party["digit"] = consti_party["digit"].astype("int64")
    consti_party = consti_party[(consti_party["digit"] >= 1) & (consti_party["digit"] <= 9)]
    per_party_digits.append(consti_party[["พรรค", "digit"]].rename(columns={"พรรค": "party"}))

if "party_name_clean" in partylist_df.columns and "First_digit" in partylist_df.columns:
    partylist_party = partylist_df[["party_name_clean", "First_digit"]].copy()
    partylist_party["digit"] = pd.to_numeric(partylist_party["First_digit"], errors="coerce")
    partylist_party = partylist_party.dropna(subset=["party_name_clean", "digit"])
    partylist_party["digit"] = partylist_party["digit"].astype("int64")
    partylist_party = partylist_party[(partylist_party["digit"] >= 1) & (partylist_party["digit"] <= 9)]
    per_party_digits.append(partylist_party[["party_name_clean", "digit"]].rename(columns={"party_name_clean": "party"}))

party_data = {}
if per_party_digits:
    merged_party_digits = pd.concat(per_party_digits, ignore_index=True)
    for party, group in merged_party_digits.groupby("party", sort=True):
        party_data[str(party)] = {
            "total": int(len(group)),
            "distribution": benford_distribution(group["digit"]),
        }

payload = {
    "sources": ["src/consti1.csv", "src/partylist1.csv"],
    "overall": {
        "total": int(len(overall_digits)),
        "distribution": benford_distribution(overall_digits),
    },
    "parties": party_data,
}

output_path = repo_root / "src" / "benford.json"
output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

output_path